# Launch GPU Jupyter and OpenCode on Chameleon

This notebook reserves a `g1.h100.pci.1` GPU VM at KVM@TACC, boots it from a disposable volume, creates S3 credentials, and starts GPU-enabled JupyterLab and OpenCode Web. Run the cells in order in the Chameleon Jupyter environment.

## Configuration

Set the lease duration and disposable boot-volume size. The default volume size is 100 GiB.

In [ ]:
LEASE_DAYS = 1
VOLUME_SIZE_GB = 100

In [ ]:
from chi import context

context.version = "1.0"
context.choose_project()

## Create S3 credentials

The credentials are created for the selected Chameleon project, cached in your Chameleon Jupyter home directory, and reused on later launches.

In [ ]:
from chi import lease, network, server
from datetime import timedelta
from pathlib import Path
from IPython.display import Markdown, display
import chi
import os
import secrets
import tempfile
import time

if not isinstance(LEASE_DAYS, (int, float)) or LEASE_DAYS <= 0:
    raise ValueError("LEASE_DAYS must be positive")
if not isinstance(VOLUME_SIZE_GB, int) or VOLUME_SIZE_GB <= 0:
    raise ValueError("VOLUME_SIZE_GB must be a positive integer")

context.choose_site(default="CHI@TACC")

conn = chi.clients.connection()
project_id = conn.current_project_id
credentials_file = Path.home() / "work" / f".jupyter-opencode-s3-{project_id}.env"

if credentials_file.exists():
    s3_env = dict(
        line.strip().split("=", 1)
        for line in credentials_file.read_text().splitlines()
        if "=" in line and not line.lstrip().startswith("#")
    )
    print(f"Reusing S3 credentials from {credentials_file}")
else:
    identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
    url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"
    resp = conn.session.post(url, json={"tenant_id": project_id})
    resp.raise_for_status()
    ec2 = resp.json()["credential"]
    s3_env = {
        "AWS_ACCESS_KEY_ID": ec2["access"],
        "AWS_SECRET_ACCESS_KEY": ec2["secret"],
        "AWS_DEFAULT_REGION": "RegionOne",
        "AWS_ENDPOINT_URL": "https://chi.tacc.chameleoncloud.org:7480",
        "AWS_ENDPOINT_URL_S3": "https://chi.tacc.chameleoncloud.org:7480",
        "S3_ENDPOINT_URL": "https://chi.tacc.chameleoncloud.org:7480",
    }
    credentials_file.parent.mkdir(parents=True, exist_ok=True)
    credentials_file.write_text("".join(f"{key}={value}\n" for key, value in s3_env.items()))
    credentials_file.chmod(0o600)
    print(f"Created S3 credentials in {credentials_file}")

credentials_file.chmod(0o600)
required_s3_keys = {"AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "S3_ENDPOINT_URL"}
if not required_s3_keys.issubset(s3_env):
    raise RuntimeError(f"S3 credential file is missing: {required_s3_keys - set(s3_env)}")
print(f"S3 access key: {s3_env['AWS_ACCESS_KEY_ID']}")

## Reserve the H100 VM

Create an immediate lease for one `g1.h100.pci.1` VM. If no H100 is available for the requested duration, use the KVM@TACC flavor calendar to find an available time.

In [ ]:
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
RESOURCE_PREFIX = f"jupyter-opencode-gpu-{username}"
LEASE_NAME = f"lease-{RESOURCE_PREFIX}"
SERVER_NAME = f"node-{RESOURCE_PREFIX}"
BOOT_VOLUME_NAME = f"boot-vol-{RESOURCE_PREFIX}"
FLAVOR = "g1.h100.pci.1"
IMAGE = "CC-Ubuntu24.04-CUDA"

l = lease.Lease(LEASE_NAME, duration=timedelta(days=LEASE_DAYS))
l.add_flavor_reservation(id=server.get_flavor_id(FLAVOR), amount=1)
l.submit(idempotent=True)
l.show()

## Create a disposable boot volume and launch the VM

In [ ]:
existing_servers = [item for item in server.list_servers() if item.name == SERVER_NAME]

if existing_servers:
    s = existing_servers[0]
    if s.status != "ACTIVE":
        raise RuntimeError(f"Server {SERVER_NAME} is {s.status}; run cleanup.ipynb before launching again")
    print(f"Reusing server {SERVER_NAME} ({s.status})")
else:
    os_conn = chi.clients.connection()
    cinder_client = chi.clients.cinder()
    images = list(os_conn.image.images(name=IMAGE))
    if not images:
        raise RuntimeError(f"Image not found: {IMAGE}")

    existing_volumes = [v for v in cinder_client.volumes.list() if v.name == BOOT_VOLUME_NAME]
    if existing_volumes:
        boot_vol = cinder_client.volumes.get(existing_volumes[0].id)
        if boot_vol.status != "available" or boot_vol.size != VOLUME_SIZE_GB:
            raise RuntimeError(f"Existing boot volume is {boot_vol.status} with size {boot_vol.size}; run cleanup.ipynb")
    else:
        boot_vol = cinder_client.volumes.create(
            name=BOOT_VOLUME_NAME,
            size=VOLUME_SIZE_GB,
            imageRef=images[0].id,
            metadata={"deployment": RESOURCE_PREFIX, "image": IMAGE},
        )

    deadline = time.time() + 1800
    while True:
        boot_vol = cinder_client.volumes.get(boot_vol.id)
        if boot_vol.status == "available":
            break
        if boot_vol.status in {"error", "error_restoring", "error_extending"}:
            raise RuntimeError(f"Boot volume provisioning failed with status {boot_vol.status}")
        if time.time() >= deadline:
            raise TimeoutError("Timed out waiting for the boot volume")
        time.sleep(10)

    bdm = [{
        "boot_index": 0,
        "uuid": boot_vol.id,
        "source_type": "volume",
        "destination_type": "volume",
        "delete_on_termination": True,
    }]
    keypair = server.update_keypair()
    server_from_vol = os_conn.compute.create_server(
        name=SERVER_NAME,
        flavor_id=server.get_flavor_id(l.get_reserved_flavors()[0].name),
        block_device_mapping_v2=bdm,
        networks=[{"uuid": os_conn.network.find_network("sharednet1").id}],
        key_name=keypair.name,
        metadata={"boot_volume_id": boot_vol.id, "lease_name": LEASE_NAME},
    )
    os_conn.compute.wait_for_server(server_from_vol, wait=1200)
    s = server.get_server(SERVER_NAME)

## Allow SSH, Jupyter, and OpenCode

Create project security groups for TCP ports 22, 8888, and 4096, then attach them to the VM.

In [ ]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH on TCP port 22"},
    {"name": "allow-jupyter", "port": 8888, "description": "Enable Jupyter on TCP port 8888"},
    {"name": "allow-opencode", "port": 4096, "description": "Enable OpenCode on TCP port 4096"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

## Assign a floating IP

In [ ]:
s.refresh()
if not s.get_floating_ip():
    s.associate_floating_ip()
s.refresh()
s.check_connectivity()
floating_ip = s.get_floating_ip()
print(f"Floating IP: {floating_ip}")

## Install Docker and NVIDIA Container Toolkit

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("""
set -e
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | \
  sudo gpg --dearmor --yes -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
  sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
  sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list >/dev/null
sudo apt-get update
sudo apt-get install -y jq nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=docker
sudo jq 'if has("exec-opts") then . else . + {"exec-opts": ["native.cgroupdriver=cgroupfs"]} end' /etc/docker/daemon.json | sudo tee /etc/docker/daemon.json.tmp >/dev/null
sudo mv /etc/docker/daemon.json.tmp /etc/docker/daemon.json
sudo systemctl restart docker
sudo docker run --rm --gpus all ubuntu nvidia-smi
""")

## Upload the service files

In [ ]:
NOTEBOOK_DIR = Path.cwd()
REMOTE_DIR = "/home/cc/chi-jupyter-opencode"
deployment_files = [".dockerignore", "Dockerfile", "compose.yaml", "compose.gpu.yaml", "start-services.sh"]
missing_files = [name for name in deployment_files if not (NOTEBOOK_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing deployment files next to this notebook: {missing_files}")

s.execute(f"mkdir -p {REMOTE_DIR}/jupyter-data/project")
for name in deployment_files:
    s.upload(str(NOTEBOOK_DIR / name), f"{REMOTE_DIR}/{name}")
print("Uploaded Docker and Compose files")

## Configure and start Jupyter and OpenCode

The project `.env` contains the S3 credentials and is ignored by Git. A separate deployment `.env` contains the browser credentials used by Compose.

In [ ]:
jupyter_token = secrets.token_hex(24)
opencode_password = secrets.token_hex(24)

deployment_env = f"""JUPYTER_PORT=8888
OPENCODE_PORT=4096
BIND_ADDRESS=0.0.0.0
JUPYTER_DATA_DIR=./jupyter-data
OPENCODE_PROJECT_DIR=project
JUPYTER_TOKEN={jupyter_token}
OPENCODE_SERVER_USERNAME=opencode
OPENCODE_SERVER_PASSWORD={opencode_password}
"""
project_env = "".join(f"{key}={value}\n" for key, value in s3_env.items())

with tempfile.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write(deployment_env)
    deployment_env_file = Path(f.name)
with tempfile.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write(project_env)
    project_env_file = Path(f.name)

deployment_env_file.chmod(0o600)
project_env_file.chmod(0o600)
s.upload(str(deployment_env_file), f"{REMOTE_DIR}/.env")
s.upload(str(project_env_file), f"{REMOTE_DIR}/jupyter-data/project/.env")
deployment_env_file.unlink()
project_env_file.unlink()

s.execute(f"chmod 600 {REMOTE_DIR}/.env {REMOTE_DIR}/jupyter-data/project/.env")
s.execute(f"cd {REMOTE_DIR} && sudo docker compose -f compose.yaml -f compose.gpu.yaml up --build -d")

## Verify the GPU, services, and object-storage access

In [ ]:
s.execute(f"""
set -e
cd {REMOTE_DIR}
for attempt in $(seq 1 60); do
    if curl -fsS 'http://127.0.0.1:8888/api/contents?token={jupyter_token}' >/dev/null \
       && curl -fsS -u 'opencode:{opencode_password}' http://127.0.0.1:4096/ >/dev/null; then
        break
    fi
    if [ "$attempt" -eq 60 ]; then
        sudo docker compose -f compose.yaml -f compose.gpu.yaml logs
        exit 1
    fi
    sleep 10
done
sudo docker compose -f compose.yaml -f compose.gpu.yaml exec -T jupyter git -C /home/jovyan/work/project check-ignore .env
sudo docker compose -f compose.yaml -f compose.gpu.yaml exec -T jupyter python -c "import boto3, os, torch; boto3.client('s3', endpoint_url=os.environ['S3_ENDPOINT_URL']).list_buckets(); print('S3 credentials verified'); print('GPU:', torch.cuda.get_device_name(0))"
sudo docker compose -f compose.yaml -f compose.gpu.yaml ps
""")

## Open the services

In [ ]:
display(Markdown(f"""
### GPU environment ready

- [Open JupyterLab](http://{floating_ip}:8888/lab?token={jupyter_token})
- [Open OpenCode](http://{floating_ip}:4096)
- OpenCode username: `opencode`
- OpenCode password: `{opencode_password}`
- SSH: `ssh cc@{floating_ip}`
- Lease: `{LEASE_NAME}` for {LEASE_DAYS} days

> These browser links use unencrypted HTTP.
"""))